# A4. 差评批量分析 Notebook

> **配套模块**: [A4 客服与售后](../paths/a-operators/a4-customer-service.md)
>
> **功能**: 上传差评 CSV → 自动分类 + 频率统计 + 改善建议 + 可视化
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a4-negative-review-analysis.ipynb)

---

## 1. 安装依赖

In [ ]:
!pip install -q pandas numpy plotly wordcloud matplotlib

## 2. 加载差评数据

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# === 选项 1: 上传你的 CSV ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# === 选项 2: 示例数据（100 条差评）===
np.random.seed(42)
neg_titles = [
    'Battery dies in 2 hours', 'Bluetooth keeps disconnecting', 'Broke after one week',
    'Terrible sound quality', 'Does not fit my ears', 'Charging case stopped working',
    'Not noise cancelling at all', 'Left earbud stopped working', 'Uncomfortable after 30 min',
    'Worse than my old earbuds', 'Returned immediately', 'False advertising on battery life',
    'Connection drops every 5 minutes', 'Cheap plastic build', 'One side louder than other'
]
neg_bodies = [
    'Battery only lasts 2 hours, not 8 as advertised. Complete waste of money. Had to return.',
    'The bluetooth connection drops every few minutes. Tried with iPhone and Android, same issue.',
    'The charging case lid broke after one week. Cheap plastic construction. Very disappointed.',
    'Sound quality is tinny with no bass at all. My $15 earbuds sound better than these.',
    'Tried all 4 ear tip sizes but none fit properly. They keep falling out during exercise.',
    'Left earbud completely stopped working after 3 weeks. No response from customer service.',
    'The noise cancellation is basically non-existent. Can still hear everything around me.',
    'After 30 minutes of wearing, my ears start hurting. The design is not ergonomic at all.',
    'Charging case takes forever to charge and the LED indicator is confusing.',
    'Audio delay when watching videos. Lip sync is off by about half a second.'
]

n = 100
df = pd.DataFrame({
    'rating': np.random.choice([1, 2], n, p=[0.4, 0.6]),
    'title': [neg_titles[i % len(neg_titles)] for i in range(n)],
    'body': [neg_bodies[i % len(neg_bodies)] for i in range(n)],
    'date': pd.date_range('2025-06-01', periods=n, freq='2D'),
    'helpful_votes': np.random.poisson(3, n)
})

df['full_text'] = df['title'] + '. ' + df['body']
print(f'Loaded {len(df)} negative reviews (1-2 stars)')
print(f'Rating distribution: {df["rating"].value_counts().to_dict()}')
df.head()

## 3. 关键词提取与问题分类

In [ ]:
# 定义问题类别和关键词
ISSUE_CATEGORIES = {
    '🔋 电池续航': ['battery', 'charge', 'dies', 'hours', 'drain', 'last'],
    '📶 蓝牙连接': ['bluetooth', 'connection', 'disconnect', 'drops', 'pair', 'connect'],
    '🔨 做工质量': ['broke', 'broken', 'cheap', 'plastic', 'build', 'quality', 'fell apart'],
    '🔊 音质问题': ['sound', 'audio', 'bass', 'tinny', 'volume', 'louder', 'delay'],
    '👂 佩戴舒适': ['fit', 'comfortable', 'ear', 'pain', 'fall', 'size', 'tip'],
    '🔇 降噪效果': ['noise', 'cancellation', 'anc', 'cancel'],
    '📦 充电盒': ['case', 'charging case', 'lid', 'led', 'indicator'],
    '🛠️ 功能故障': ['stopped working', 'defective', 'malfunction', 'dead']
}

def classify_issue(text):
    text_lower = text.lower()
    matches = []
    for category, keywords in ISSUE_CATEGORIES.items():
        if any(kw in text_lower for kw in keywords):
            matches.append(category)
    return matches if matches else ['❓ 其他']

df['issues'] = df['full_text'].apply(classify_issue)
df['primary_issue'] = df['issues'].apply(lambda x: x[0])

# 统计
issue_counts = df.explode('issues')['issues'].value_counts()
print('=== 差评问题分类排名 ===')
for issue, count in issue_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'{issue}: {count} ({pct:.0f}%) {bar}')

## 4. 可视化

In [ ]:
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 问题分布饼图
fig = px.pie(values=issue_counts.values, names=issue_counts.index,
             title='差评问题分布', hole=0.3)
fig.show()

# 问题趋势（月度）
df['month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
exploded = df.explode('issues')
monthly_issues = exploded.groupby(['month', 'issues']).size().reset_index(name='count')
fig = px.line(monthly_issues, x='month', y='count', color='issues',
              title='差评问题月度趋势（哪些问题在恶化？）')
fig.show()

# 词云
all_text = ' '.join(df['full_text'].tolist())
wc = WordCloud(width=800, height=400, background_color='white', max_words=60, colormap='Reds').generate(all_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('差评关键词词云', fontsize=16)
plt.show()

## 5. 生成改善建议

In [ ]:
print('=== 产品改善优先级建议 ===')
print()
for i, (issue, count) in enumerate(issue_counts.head(5).items(), 1):
    pct = count / len(df) * 100
    severity = '🔴 紧急' if pct > 20 else ('🟡 重要' if pct > 10 else '🟢 关注')
    print(f'{i}. {issue} — {count} 条差评 ({pct:.0f}%) [{severity}]')
    # 显示该类别的典型差评
    examples = df[df['primary_issue'] == issue]['full_text'].head(2)
    for ex in examples:
        print(f'   → "{ex[:80]}..."')
    print()

print('\n=== 行动建议 ===')
print('P0 (本周): 解决排名第 1 的问题 — 直接影响退货率和评分')
print('P1 (本月): 解决排名第 2-3 的问题 — 在 Listing 中管理预期')
print('P2 (下月): 在 Q&A 中预埋排名前 5 问题的回答（Rufus 优化）')

## 6. 导出

In [ ]:
# 导出带分类的差评数据
df[['date', 'rating', 'title', 'body', 'primary_issue', 'helpful_votes']].to_csv('negative_reviews_classified.csv', index=False)

# 导出问题摘要
summary = issue_counts.reset_index()
summary.columns = ['Issue', 'Count']
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(1)
summary.to_csv('issue_summary.csv', index=False)

print('Exported: negative_reviews_classified.csv, issue_summary.csv')